<a href="https://colab.research.google.com/github/SofiiaBobr/goit_machine_learning/blob/main/Numerical_Programming/iris_spectral_clustering_bobrivets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнє завдання: Спектральна кластеризація на наборі даних Iris

**Курс:** Numerical Programming in Python  
**Тема 1:** Власні значення та вектори

У цьому ноутбуці виконується аналіз набору даних Iris, стандартизація ознак, спектральна кластеризація та оцінка результатів за допомогою Confusion Matrix.

## 1. Імпорт бібліотек

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import SpectralClustering
from sklearn.metrics import confusion_matrix, adjusted_rand_score, normalized_mutual_info_score
from scipy.optimize import linear_sum_assignment

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

## 2. Завантаження даних Iris та створення DataFrame

Використаємо `load_iris()` з бібліотеки `sklearn`, а потім створимо `pandas.DataFrame`.

In [ ]:
iris = load_iris()

df = pd.DataFrame(iris.data, columns=iris.feature_names)
df["target"] = iris.target
df["class"] = df["target"].map({i: name for i, name in enumerate(iris.target_names)})

df.head()

## 3. Базова інформація про дані

In [ ]:
print("Shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

df.describe()

## 4. Розподіл спостережень за класами

Перевіримо, скільки спостережень належить до кожного класу.

In [ ]:
sns.countplot(data=df, x="class")
plt.title("Distribution of Iris classes")
plt.xlabel("Class")
plt.ylabel("Count")
plt.xticks(rotation=20)
plt.show()

df["class"].value_counts()

## 5. Візуалізація ознак

Побудуємо pairplot, щоб побачити, як класи розділяються у просторі ознак.

In [ ]:
sns.pairplot(df, vars=iris.feature_names, hue="class", diag_kind="hist")
plt.show()

## 6. Підготовка даних та стандартизація

Для кластеризації використаємо тільки числові ознаки. Оскільки ознаки мають різні масштаби, застосуємо `StandardScaler`.

In [ ]:
X = df[iris.feature_names].values
y_true = df["target"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pd.DataFrame(X_scaled, columns=iris.feature_names).describe()

## 7. Спектральна кластеризація

Оскільки в наборі Iris є 3 класи, встановимо `n_clusters=3`. Спочатку використаємо affinity `nearest_neighbors`.

In [ ]:
spectral = SpectralClustering(
    n_clusters=3,
    affinity="nearest_neighbors",
    n_neighbors=10,
    assign_labels="kmeans",
    random_state=42
)

clusters = spectral.fit_predict(X_scaled)

df["cluster"] = clusters
df[["class", "target", "cluster"]].head(10)

## 8. Порівняння кластерів із реальними класами

Кластери не мають наперед заданих назв, тому номер кластера може не збігатися з номером класу.  
Спочатку побудуємо confusion matrix у сирому вигляді.

In [ ]:
cm = confusion_matrix(y_true, clusters)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix: true classes vs predicted clusters")
plt.xlabel("Predicted cluster")
plt.ylabel("True class")
plt.show()

cm

## 9. Узгодження номерів кластерів із класами

Щоб коректніше інтерпретувати результат, зіставимо кожен кластер із найбільш відповідним класом за допомогою алгоритму Hungarian matching.

In [ ]:
def map_clusters_to_classes(y_true, clusters):
    cm = confusion_matrix(y_true, clusters)
    row_ind, col_ind = linear_sum_assignment(-cm)
    mapping = {cluster: true_class for true_class, cluster in zip(row_ind, col_ind)}
    mapped_clusters = np.array([mapping[c] for c in clusters])
    return mapped_clusters, mapping

mapped_clusters, mapping = map_clusters_to_classes(y_true, clusters)

print("Cluster to class mapping:", mapping)

cm_mapped = confusion_matrix(y_true, mapped_clusters)

plt.figure(figsize=(6, 4))
sns.heatmap(
    cm_mapped,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=iris.target_names,
    yticklabels=iris.target_names
)
plt.title("Confusion Matrix after cluster mapping")
plt.xlabel("Predicted class after mapping")
plt.ylabel("True class")
plt.show()

cm_mapped

## 10. Метрики якості кластеризації

In [ ]:
accuracy = np.trace(cm_mapped) / np.sum(cm_mapped)
ari = adjusted_rand_score(y_true, clusters)
nmi = normalized_mutual_info_score(y_true, clusters)

print(f"Accuracy after mapping: {accuracy:.4f}")
print(f"Adjusted Rand Index: {ari:.4f}")
print(f"Normalized Mutual Information: {nmi:.4f}")

## 11. Візуалізація результатів кластеризації

Для 2D-візуалізації використаємо дві ознаки: `petal length` та `petal width`, бо вони добре розділяють класи Iris.

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
sns.scatterplot(
    data=df,
    x="petal length (cm)",
    y="petal width (cm)",
    hue="class",
    palette="deep"
)
plt.title("True Iris classes")

plt.subplot(1, 2, 2)
sns.scatterplot(
    data=df,
    x="petal length (cm)",
    y="petal width (cm)",
    hue="cluster",
    palette="deep"
)
plt.title("Spectral clustering result")

plt.tight_layout()
plt.show()

## 12. Експерименти з параметрами спектральної кластеризації

Перевіримо, як змінюється результат при різних значеннях `n_neighbors`.

In [ ]:
experiment_results = []

for n_neighbors in [3, 5, 10, 15, 20, 30]:
    model = SpectralClustering(
        n_clusters=3,
        affinity="nearest_neighbors",
        n_neighbors=n_neighbors,
        assign_labels="kmeans",
        random_state=42
    )
    pred = model.fit_predict(X_scaled)
    mapped_pred, _ = map_clusters_to_classes(y_true, pred)
    cm_exp = confusion_matrix(y_true, mapped_pred)
    acc = np.trace(cm_exp) / np.sum(cm_exp)
    ari_score = adjusted_rand_score(y_true, pred)
    nmi_score = normalized_mutual_info_score(y_true, pred)

    experiment_results.append({
        "n_neighbors": n_neighbors,
        "accuracy": acc,
        "ARI": ari_score,
        "NMI": nmi_score
    })

experiment_df = pd.DataFrame(experiment_results)
experiment_df

In [ ]:
plt.plot(experiment_df["n_neighbors"], experiment_df["accuracy"], marker="o", label="Accuracy")
plt.plot(experiment_df["n_neighbors"], experiment_df["ARI"], marker="o", label="ARI")
plt.plot(experiment_df["n_neighbors"], experiment_df["NMI"], marker="o", label="NMI")
plt.xlabel("n_neighbors")
plt.ylabel("Score")
plt.title("Spectral clustering quality for different n_neighbors")
plt.legend()
plt.grid(True)
plt.show()

## 13. Додатковий експеримент: affinity='rbf'

Також перевіримо варіант із RBF affinity. Для цього поекспериментуємо з параметром `gamma`.

In [ ]:
rbf_results = []

for gamma in [0.1, 0.5, 1, 2, 5, 10]:
    model = SpectralClustering(
        n_clusters=3,
        affinity="rbf",
        gamma=gamma,
        assign_labels="kmeans",
        random_state=42
    )
    pred = model.fit_predict(X_scaled)
    mapped_pred, _ = map_clusters_to_classes(y_true, pred)
    cm_exp = confusion_matrix(y_true, mapped_pred)
    acc = np.trace(cm_exp) / np.sum(cm_exp)
    ari_score = adjusted_rand_score(y_true, pred)
    nmi_score = normalized_mutual_info_score(y_true, pred)

    rbf_results.append({
        "gamma": gamma,
        "accuracy": acc,
        "ARI": ari_score,
        "NMI": nmi_score
    })

rbf_df = pd.DataFrame(rbf_results)
rbf_df

In [ ]:
plt.plot(rbf_df["gamma"], rbf_df["accuracy"], marker="o", label="Accuracy")
plt.plot(rbf_df["gamma"], rbf_df["ARI"], marker="o", label="ARI")
plt.plot(rbf_df["gamma"], rbf_df["NMI"], marker="o", label="NMI")
plt.xlabel("gamma")
plt.ylabel("Score")
plt.title("Spectral clustering quality for different gamma values")
plt.legend()
plt.grid(True)
plt.show()

## Висновки

1. Набір даних Iris містить 150 спостережень, 4 числові ознаки та 3 реальні класи: setosa, versicolor і virginica.
2. Перед кластеризацією дані були стандартизовані за допомогою `StandardScaler`, що важливо для методів, які залежать від відстаней між об'єктами.
3. Спектральна кластеризація дозволяє розділити дані Iris на 3 кластери без використання міток класів під час навчання.
4. Клас setosa зазвичай відокремлюється найкраще, оскільки він добре відділений від інших класів у просторі ознак.
5. Найбільші помилки виникають між класами versicolor та virginica, бо ці класи частково перетинаються за ознаками.
6. Confusion Matrix показує, наскільки отримані кластери відповідають реальним класам після зіставлення номерів кластерів із класами.
7. Експерименти з параметрами `n_neighbors` та `gamma` показують, що якість спектральної кластеризації залежить від вибору параметрів affinity.
8. Отже, спектральна кластеризація є корисним методом для пошуку структури в даних, але результат потрібно оцінювати та інтерпретувати з урахуванням параметрів моделі.